# Movie Review Sentiment Analysis (spaCy + TF-IDF + Naive Bayes)

In [23]:
import pandas as pd
import spacy
import pickle

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [25]:
df = pd.read_csv(r"C:\Users\sachin-selvam\Documents\movie review\IMDB_Dataset_CLEANED.csv")

print("Dataset Shape:", df.shape)

Dataset Shape: (49396, 2)


In [26]:
print(df["sentiment"].value_counts())

sentiment
negative    24698
positive    24698
Name: count, dtype: int64


In [27]:
nlp = spacy.load(
    "en_core_web_sm",
    disable=["parser", "ner"]
)

print("spaCy Model Loaded")

spaCy Model Loaded


In [28]:
clean_reviews = []

for doc in nlp.pipe(
    df["review"].astype(str),
    batch_size=500
):
    tokens = [
        token.lemma_
        for token in doc
        if not token.is_stop
        and not token.is_punct
        and not token.is_space
    ]

    clean_reviews.append(" ".join(tokens))

df["clean_review"] = clean_reviews

print("Preprocessing Completed")

Preprocessing Completed


In [29]:
df[["review", "clean_review"]].head()

,review,clean_review
0,"$25,000 Pyramid Clues: Deep Blue Sea. Tremors....","$ 25,000 Pyramid clue Deep Blue Sea tremor Sli..."
1,0.5/10. This movie has absolutely nothing good...,0.5/10 movie absolutely good acting bad see am...
2,"0*'s Christian Slater, Tara Reid, Stephen Dorf...",0 Christian Slater Tara Reid Stephen Dorff Fra...
3,"102 Dalmatians (2000, Dir. Kevin Lima) <br /><...",102 Dalmatians 2000 Dir Kevin Lima < br /><br ...
4,102 DALMATIANS [Walt Disney]: I wasn't a fan o...,102 dalmatian Walt Disney fan previous install...


In [31]:
tfidf = TfidfVectorizer(max_features=5000)

X = tfidf.fit_transform(df["clean_review"])

y = df["sentiment"]

print("TF-IDF Completed")

TF-IDF Completed


In [32]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Train Shape:", X_train.shape)
print("Test Shape:", X_test.shape)

Train Shape: (39516, 5000)
Test Shape: (9880, 5000)


In [33]:
model.fit(X_train, y_train)

print("Model Training Completed")

Model Training Completed


In [34]:
y_pred = model.predict(X_test)

In [35]:
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", round(accuracy * 100, 2), "%")

Accuracy: 85.2 %


In [36]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

    negative       0.86      0.84      0.85      4940
    positive       0.84      0.86      0.85      4940

    accuracy                           0.85      9880
   macro avg       0.85      0.85      0.85      9880
weighted avg       0.85      0.85      0.85      9880



In [37]:
print(confusion_matrix(y_test, y_pred))

[[4157  783]
 [ 679 4261]]


In [38]:
pickle.dump(model, open("model.pkl", "wb"))
pickle.dump(tfidf, open("tfidf.pkl", "wb"))

print("Model Saved Successfully")

Model Saved Successfully


In [39]:
def preprocess(text):
    
    doc = nlp(text.lower())

    tokens = [
        token.lemma_
        for token in doc
        if not token.is_stop
        and not token.is_punct
        and not token.is_space
    ]

    return " ".join(tokens)

In [40]:
review = input("Enter Movie Review: ")

In [41]:
clean_review = preprocess(review)

review_vector = tfidf.transform([clean_review])

prediction = model.predict(review_vector)[0]

print("Predicted Sentiment:", prediction)

Predicted Sentiment: negative
